In [45]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


In [46]:
# 데이터 로드 및 초기 정제

query = 'SELECT nct_id, name FROM ctgov.conditions; '
df_conditions = pd.read_sql(query, engine)

### conditions table
- 수천개의 질환 이름을 해석 가능하고 모델에서 바로 사용할 수 있는 질환 범주로 통합

In [47]:
df_conditions.head()

,nct_id,name
0,NCT05201144,Congenital Diaphragmatic Hernia
1,NCT05201144,Pulmonary Hypertension
2,NCT02017717,Recurrent Glioblastoma
3,NCT04553757,Brain Neoplasm
4,NCT04553757,Low Grade Glioma


In [48]:
# 소문자로 통일
df_conditions['name'] = df_conditions['name'].str.lower()
df_conditions['name'].unique()

<StringArray>
[                        'congenital diaphragmatic hernia',
                                  'pulmonary hypertension',
                                  'recurrent glioblastoma',
                                          'brain neoplasm',
                                        'low grade glioma',
                                        'seizure disorder',
                      'decision support systems, clinical',
                                           'heart failure',
                                          'batten disease',
                                                    'cln2',
 ...
                             'children with heart defects',
                                  'phagocytic dysfunction',
         'epstein-barr virus-associated gastric carcinoma',
 'burn any degree involving 20-29 percent of body surface',
 'burn any degree involving 30-39 percent of body surface',
 'burn any degree involving 40-49 percent of body surface',
 'burn any degree inv

In [49]:
# df_studies 파일 불러오기
load_path = r'C:\dev\clinicaltrials-study\data\processed\studies_clean.csv'
df_studies = pd.read_csv(load_path)
df_studies.head()

,nct_id,study_type,enrollment,overall_status,number_of_arms,has_dmc,has_expanded_access,is_fda_regulated_drug,is_fda_regulated_device,duration_of_study,phase_1,phase_2,phase_3,phase_4,phase_unknown
0,NCT00946062,INTERVENTIONAL,190.0,COMPLETED,2.0,0,0,-1,-1,973,0,0,0,0,1
1,NCT03964701,OBSERVATIONAL,18.0,UNKNOWN,2.0,-1,0,0,0,366,0,0,0,0,1
2,NCT06515847,INTERVENTIONAL,60.0,COMPLETED,2.0,0,0,0,0,365,0,0,0,0,1
3,NCT03717220,INTERVENTIONAL,26.0,COMPLETED,1.0,-1,0,0,0,2937,0,0,0,0,1
4,NCT03159455,INTERVENTIONAL,48.0,COMPLETED,2.0,0,0,0,0,192,1,0,0,0,0


In [50]:
print(df_studies.shape)

(557402, 15)


- interventional (중재연구): 약물이나 치료법을 실제로 적용하고 효과를 보는 연구. 
- observational (관찰연구): 개입 없이 현상을 지켜보는 연구로, 성공/실패의 개념이 다르기 때문에 제외

In [51]:
# 파일 합치기
df_conditions = df_conditions.merge(df_studies[['nct_id', 'study_type']], on='nct_id', how='left')
# 데이터 필터링 - interventional만
df_conditions = df_conditions[df_conditions['study_type']=='INTERVENTIONAL']
# 불필요한 컬럼 제거
df_conditions.drop(columns=['study_type'], inplace=True)
df_conditions.head()

,nct_id,name
0,NCT05201144,congenital diaphragmatic hernia
1,NCT05201144,pulmonary hypertension
2,NCT02017717,recurrent glioblastoma
6,NCT06847906,"decision support systems, clinical"
7,NCT06847906,heart failure


In [52]:
print(df_conditions.shape)

(745810, 2)


In [53]:
df_conditions.info()

<class 'pandas.DataFrame'>
Index: 745810 entries, 0 to 1023640
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   nct_id  745810 non-null  str  
 1   name    745810 non-null  str  
dtypes: str(2)
memory usage: 17.1 MB


In [54]:
# 질환명 그룹핑 전에 핵심 질병 추출
top_conditions = df_conditions['name'].value_counts().head(1000)
top_conditions.to_csv('file/conditions_top1000.csv')

In [55]:
# 수동으로 그룹핑한 파일 불러와서 매핑하기 (파생변수 생성)

conditions_1000 = pd.read_csv('file/conditions_top1000_grouped.csv')
mapping_dict = dict(zip(conditions_1000['name'], conditions_1000['grouped_conditions']))
df_conditions['grouped_conditions'] = df_conditions['name'].map(mapping_dict)

In [56]:
df_conditions['name'].value_counts()

name
healthy                                                    8944
breast cancer                                              6203
obesity                                                    5755
stroke                                                     3848
depression                                                 3720
                                                           ... 
burn any degree involving 40-49 percent of body surface       1
burn any degree involving 50-59 percent of body surface       1
burn any degree involving 60-65 percent of body surface       1
cnlc iiia stage                                               1
dermatology disease                                           1
Name: count, Length: 98653, dtype: int64

In [57]:
df_conditions['grouped_conditions'].value_counts()

grouped_conditions
cancers                          78461
others                           48720
endocrine/metabolic_disorders    28324
mental_health_disorders          28217
cardiovascular_diseases          26329
neurological_disorders           24912
infectious_diseases              17701
pain_disorders                   14158
healthy_participants             13670
respiratory_disorders            13257
musculoskeletal_disorders        11181
gastrointestinal_disorders       10494
reproductive_disorders            7207
opthamological_disorders          6744
renal/urological_disorders        6183
dermatological_disorders          4613
neurodevelopmental_disorders      2943
dental_disorders                  2769
infectious_disorders               169
Name: count, dtype: int64

In [58]:
df_conditions.info()

<class 'pandas.DataFrame'>
Index: 745810 entries, 0 to 1023640
Data columns (total 3 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   nct_id              745810 non-null  str  
 1   name                745810 non-null  str  
 2   grouped_conditions  346052 non-null  str  
dtypes: str(3)
memory usage: 22.8 MB


In [59]:
# 나머지 매핑되지 않은 질환 이름 분류하는 키워드 사전
keyword_map = {
    "cancers": ["astrocytoma", "metastases", "leukemia", "lymphoma", "malignancy", "melanoma", "mds", "myelodysplastic",
                "myeloproliferative", "cancer", "carcinoma", "tumor", "neoplasm", "sarcoma", "glioblastoma", "glioma"],
    "cardiovascular_diseases": ["cardio", "arrhythmia", "atrial", "cardiac", "vein", "ischemia", "myocardial", "arterial", "vascular", "heart", "coronary", "aortic", "artery", "stroke", "hypertension", "angina"],
    "dental_disorders": ["dental", "tooth", "caries", "periodontitis", "oral hygiene", "gingivitis"],
    "dermatological_disorders": ["acne", "psoriasis", "eczema", "rash", "skin", "derm"],
    "endocrine/metabolic_disorders": ["diabetes", "thyroid", "insulin", "metabolic", "endocrine", "glucose", "obesity", "overweight", "parathyroidism"],
    "gastrointestinal_disorders": ["gastro", "liver", "colon", "bowel", "hepatitis", "ulcer", "esophageal", "pancreas", "celiac", "hernia", "colitis", "crohn"],
    "genetic_disorders": ["genetic", "mutation", "chromosome", "inherited", "congenital"],
    "hepatic_disorders": ["cirrhosis", "liver", "hepatic", "nafld"],
    "infectious_diseases": ["corona", "sars", "immunodeficiency", "influenza", "sepsis", "tuberculosis", "tb", "std", "hiv", "viral", "bacterial", "infection", "covid", "hepatitis"],
    "mental_health_disorders": ["delirium", "mental disorders", "ocd", "compulsive", "stress", "insomnia", "psychosis", "schizophrenia", "abuse", "dependence", "use disorders", "depression", "anxiety", "ptsd", 
                         "bipolar", "autism", "adhd"],
    "musculoskeletal_disorders": ["arthritis", "disc", "fracture", "dystrophy", "myalgia", "gout", "osteo", "osteoporosis", "joint", "bone", "muscle", "capsulitis", "back pain"],
    "neurological_disorders": ["neuro", "brain", "sclerosis", "cerebral", "stroke", "headache", "meningitis", "migraine", "nervous", "seizure", "parkinson", "epilepsy", "dementia", "als", "ms"],
    "opthamological_disorders": ["eye", "glaucoma", "macular", "myopia", "presbyopia", "retina"],
    "pain_disorders": ["pain"],
    "respiratory_disorders": ["sinusitis", "respiratory", "bronchitis", "pulmonary", "copd", "cystic fibrosis", "pneumonia", "asthma", "lung"],
    "renal/urological_disorders": ["renal", "kidney", "urinary", "bladder", "prostate"],
    "reproductive_health": ["pregnancy", "fertility", "menstrual", "contraception", "ovarian", "uterine", "pcod","ovary", "menopause"]
}

In [60]:
# 매핑시 우선순위 지정
priority_order = list(keyword_map.keys())
# keyowrd_map으로 분류
def classify_conditions_priority(name):
    name = str(name).lower().strip()
    for group in priority_order:
        keywords = keyword_map[group]
        if any(keyword in name for keyword in keywords):
            return group
    return 'others'   # 매칭되는 키워드가 없으면 others로
# 1000개 수동매핑 결과와 우선순위 기반 키워드 분류기 하나로 합치기
df_conditions['grouped_conditions'] = df_conditions.apply(lambda row: row['grouped_conditions'] if pd.notnull(row['grouped_conditions']) else classify_conditions_priority(row['name']), axis=1)

df_conditions['grouped_conditions'].value_counts()

grouped_conditions
others                           258781
cancers                          155844
cardiovascular_diseases           44938
neurological_disorders            39028
mental_health_disorders           36927
endocrine/metabolic_disorders     35091
infectious_diseases               28591
musculoskeletal_disorders         22771
gastrointestinal_disorders        20694
respiratory_disorders             19623
pain_disorders                    18812
healthy_participants              13670
renal/urological_disorders        11098
opthamological_disorders           8961
dermatological_disorders           8324
reproductive_disorders             7207
dental_disorders                   6400
reproductive_health                3292
neurodevelopmental_disorders       2943
genetic_disorders                  2036
hepatic_disorders                   610
infectious_disorders                169
Name: count, dtype: int64

In [61]:

df_conditions['name'].value_counts()

name
healthy                                                    8944
breast cancer                                              6203
obesity                                                    5755
stroke                                                     3848
depression                                                 3720
                                                           ... 
burn any degree involving 40-49 percent of body surface       1
burn any degree involving 50-59 percent of body surface       1
burn any degree involving 60-65 percent of body surface       1
cnlc iiia stage                                               1
dermatology disease                                           1
Name: count, Length: 98653, dtype: int64

In [62]:
# 유사 카테고리 통합
df_conditions['grouped_conditions'] = df_conditions['grouped_conditions'].replace({
    'infectious_disorders': 'infectious_diseases',       #용어 통일
    'mental_health_disorders': 'mental_disorders',       # 용어 통일
    'neurological_disorders': 'mental_disorders',  # 정신건강을 심리로
    'reproductive_disorders': 'reproductive_health',     # 용어 변경(질환 뿐만 아니라 건강관리 측면까지 포괄적인 용어로)
    'hepatic_disorders': 'gastrointestinal_disorders'    # 간질환을 소화기계 범주로
})

In [63]:
df_conditions['grouped_conditions'].value_counts()

grouped_conditions
others                           258781
cancers                          155844
mental_disorders                  75955
cardiovascular_diseases           44938
endocrine/metabolic_disorders     35091
infectious_diseases               28760
musculoskeletal_disorders         22771
gastrointestinal_disorders        21304
respiratory_disorders             19623
pain_disorders                    18812
healthy_participants              13670
renal/urological_disorders        11098
reproductive_health               10499
opthamological_disorders           8961
dermatological_disorders           8324
dental_disorders                   6400
neurodevelopmental_disorders       2943
genetic_disorders                  2036
Name: count, dtype: int64

In [64]:
df_conditions.info()

<class 'pandas.DataFrame'>
Index: 745810 entries, 0 to 1023640
Data columns (total 3 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   nct_id              745810 non-null  str  
 1   name                745810 non-null  str  
 2   grouped_conditions  745810 non-null  str  
dtypes: str(3)
memory usage: 22.8 MB


In [65]:
# 원핫 인코딩
df_conditions = pd.get_dummies(df_conditions, columns=['grouped_conditions'], prefix='condt', dtype=int )
# nct_id별로 그룹핑 (한 연구에 여러 질환이 있는 경우를 위해)
df_conditions = df_conditions.groupby('nct_id').max().reset_index()
df_conditions.info()

<class 'pandas.DataFrame'>
RangeIndex: 426516 entries, 0 to 426515
Data columns (total 20 columns):
 #   Column                               Non-Null Count   Dtype
---  ------                               --------------   -----
 0   nct_id                               426516 non-null  str  
 1   name                                 426516 non-null  str  
 2   condt_cancers                        426516 non-null  int64
 3   condt_cardiovascular_diseases        426516 non-null  int64
 4   condt_dental_disorders               426516 non-null  int64
 5   condt_dermatological_disorders       426516 non-null  int64
 6   condt_endocrine/metabolic_disorders  426516 non-null  int64
 7   condt_gastrointestinal_disorders     426516 non-null  int64
 8   condt_genetic_disorders              426516 non-null  int64
 9   condt_healthy_participants           426516 non-null  int64
 10  condt_infectious_diseases            426516 non-null  int64
 11  condt_mental_disorders               426516 non-nu

In [66]:
# 불필요한 컬럼 삭제 (healthy_participants 정상군)
df_conditions.drop(columns=['name', 'condt_healthy_participants'], inplace=True, errors='ignore')


In [67]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'conditions_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_conditions.to_csv(full_save_path, index=False)

In [68]:
df_conditions.head()

,nct_id,condt_cancers,condt_cardiovascular_diseases,condt_dental_disorders,condt_dermatological_disorders,condt_endocrine/metabolic_disorders,condt_gastrointestinal_disorders,condt_genetic_disorders,condt_infectious_diseases,condt_mental_disorders,condt_musculoskeletal_disorders,condt_neurodevelopmental_disorders,condt_opthamological_disorders,condt_others,condt_pain_disorders,condt_renal/urological_disorders,condt_reproductive_health,condt_respiratory_disorders
0,NCT00000113,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,NCT00000114,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,NCT00000115,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,NCT00000116,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,NCT00000117,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


In [69]:
df_conditions.info()

<class 'pandas.DataFrame'>
RangeIndex: 426516 entries, 0 to 426515
Data columns (total 18 columns):
 #   Column                               Non-Null Count   Dtype
---  ------                               --------------   -----
 0   nct_id                               426516 non-null  str  
 1   condt_cancers                        426516 non-null  int64
 2   condt_cardiovascular_diseases        426516 non-null  int64
 3   condt_dental_disorders               426516 non-null  int64
 4   condt_dermatological_disorders       426516 non-null  int64
 5   condt_endocrine/metabolic_disorders  426516 non-null  int64
 6   condt_gastrointestinal_disorders     426516 non-null  int64
 7   condt_genetic_disorders              426516 non-null  int64
 8   condt_infectious_diseases            426516 non-null  int64
 9   condt_mental_disorders               426516 non-null  int64
 10  condt_musculoskeletal_disorders      426516 non-null  int64
 11  condt_neurodevelopmental_disorders   426516 non-nu